# 04 — Train: ARIMAX

Fit a recursive one-step `X(t) → level(t+1)` regression with ARIMA errors across all six shared feature subsets. Forecasts are fully recursive: each step feeds the previous forecast back into the target station's level predictors. The experiment uses the shared eligible cohort, expanding folds, embargo, CV selection metric, and sealed test.


In [ ]:
%load_ext autoreload
%autoreload 2

## Fixed experiment configuration

Thirteen nonseasonal orders (p,q ∈ {0,1,2} for d=0; p,q ∈ {0,1} for d=1) × six subsets × five folds = 390 candidate-fold evaluations. Each target and all predictor columns are standardized within each fold; constant and redundant columns remain. Models with `d=0` include an intercept; with `d=1` there is no intercept or drift.

The regression and ARIMA errors are fitted by iterated feasible GLS. Kalman-whitened regression and ARMA maximum likelihood alternate until relative log-likelihood change falls below `GLS_TOL`, or after `MAX_OUTER_ITER` iterations. Hourly gaps remain missing observations. Finite nonconverged fits remain eligible and their numerical diagnostics are logged. A failed or nonfinite fold marks its candidate `failed` and excludes it from selection. Exact metric ties prefer fewer features, then lower p+q, lower d, stable subset name, and ascending p,q.

Candidate processes use one numerical-library thread each and are logged as they finish; run names keep their grid position. Set `N_WORKERS = 1` for sequential search.


In [ ]:
from pathlib import Path
from time import time_ns
from uuid import uuid4

import matplotlib.pyplot as plt
import mlflow
import pandas as pd
from IPython.display import display
from mlflow.tracking import MlflowClient
from threadpoolctl import threadpool_limits
from tqdm.auto import tqdm

from src.arimax import (
    candidate_grid,
    candidate_table,
    evaluate_candidates,
    fit_arimax,
    log_cv_search,
    rolling_forecasts,
    run_params,
    save_arimax_model,
    sealed_test_histories,
    select_candidate,
)
from src.config import (
    CV_SELECTION_METRIC,
    EMBARGO_HOURS,
    FORECAST_HORIZON_HOURS,
    INITIAL_TRAIN_FRACTION,
    MLFLOW_TRACKING_URI,
    N_VALIDATION_FOLDS,
    TARGET_STATION_ID,
    WEATHER_VARIABLES,
)
from src.dataset import load_joined_dataset
from src.plots import (
    cv_error_boxplots_figure,
    predicted_vs_actual_figure,
    regime_aggregate_figure,
    regime_horizon_figure,
    test_error_boxplots_figure,
)
from src.regime_persistence import regime_mlflow_metrics, regime_mlflow_params
from src.tracking import backdated_run, metric_log_dict, run_tags, sealed_test_tables
from src.training import prediction_preview

In [ ]:
N_WORKERS = 10
MAXITER = 200
MAX_OUTER_ITER = 50
GLS_TOL = 1e-6
PREDICTION_PREVIEW_ROWS = 5
PROCESSED_DIR = Path("data/processed/joined")
METADATA_PATH = PROCESSED_DIR / "all_stations_feature_metadata.json"
TRAIN_PATH = PROCESSED_DIR / "all_stations_train_features.parquet"
TEST_PATH = PROCESSED_DIR / "all_stations_test_features.parquet"
MODEL_PATH = Path("models") / f"arimax_{TARGET_STATION_ID}.joblib"
MODEL_METADATA_PATH = MODEL_PATH.with_suffix(".json")
NOTEBOOK_EXECUTION_UUID = str(uuid4())

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
MLFLOW_EXPERIMENT = mlflow.set_experiment("arimax")
MLFLOW_CLIENT = MlflowClient()

## Validated dataset and separate histories

The loader validates the feature contract and provenance. Every subset uses the same full-contract eligible training cohort. The one-step model uses only the first target column, so its latest training label is the final training issue time plus one hour; the error state is still updated with every observed level up to each forecast origin. Predictor and observed-target histories remain separate by partition until final evaluation. Their UTC hourly grids preserve elapsed time across gaps.

A level observed through the forecast origin updates the fixed-parameter error state only when the preceding predictor row is complete. Imputed levels are treated as missing. Future-window `target_valid` flags have no role in updates.


In [ ]:
dataset = load_joined_dataset(
    METADATA_PATH,
    TRAIN_PATH,
    TEST_PATH,
    station_id=TARGET_STATION_ID,
    forecast_horizon_hours=FORECAST_HORIZON_HOURS,
    weather_variables=WEATHER_VARIABLES,
    initial_train_fraction=INITIAL_TRAIN_FRACTION,
    n_validation_folds=N_VALIDATION_FOLDS,
    embargo_rows=EMBARGO_HOURS,
)
contract = dataset.contract
TARGET_COLUMNS = list(contract.target_columns)
train_rows, test_rows = dataset.train_rows, dataset.test_rows
CANDIDATES = candidate_grid(dataset.feature_subsets)
COMMON_PARAMS = run_params(
    dataset,
    station_id=TARGET_STATION_ID,
    forecast_horizon_hours=FORECAST_HORIZON_HOURS,
    candidate_count=len(CANDIDATES),
    n_workers=N_WORKERS,
    maxiter=MAXITER,
    max_outer_iter=MAX_OUTER_ITER,
    gls_tol=GLS_TOL,
    selection_metric=CV_SELECTION_METRIC,
    initial_train_fraction=INITIAL_TRAIN_FRACTION,
    embargo_rows=EMBARGO_HOURS,
)
display(
    pd.DataFrame(
        [
            {"subset": name, "feature_count": len(columns)}
            for name, columns in dataset.feature_subsets.items()
        ]
    )
)

## Shared cross-validation with causal state updates

The model maps `X(s)` to `y(s+1)`. At issue time `t`, observed one-step residuals through `y(t)` update its error state. Forecast step `k` predicts `y(t+k)` from input hour `t+k-1` and propagates the error state without observed future levels. Step 1 uses the issue-time predictors. Later steps rebuild the target station's water-level, lag, change, rolling-statistic, and imputation-count predictors with the Stage-3 formulas (`src.feature_engineering.target_level_features`) from the levels through `t` followed by the forecasts for `t+1`…`t+k-1`, counted as not imputed, and recompute the UTC calendar. Upstream levels, weather, and rolling precipitation/temperature have no forecasts and are held at their issue-time values. No predictor row or observation after `t` is read.

Missing hours advance the state without measurement updates. The training snapshot remains unchanged. Candidate parents and nested fold runs share the execution UUID. Numerical fit diagnostics are logged as MLflow JSON artifacts; run durations use measured evaluation intervals.


In [ ]:
evaluations = log_cv_search(
    evaluate_candidates(
        CANDIDATES,
        dataset.predictor_train_history,
        dataset.target_train_history,
        train_rows,
        dataset.folds,
        contract=contract,
        n_workers=N_WORKERS,
        maxiter=MAXITER,
        max_outer_iter=MAX_OUTER_ITER,
        tol=GLS_TOL,
    ),
    candidates=CANDIDATES,
    client=MLFLOW_CLIENT,
    experiment_id=MLFLOW_EXPERIMENT.experiment_id,
    execution_uuid=NOTEBOOK_EXECUTION_UUID,
    common_params=COMMON_PARAMS,
)
comparison = candidate_table(evaluations)
selected_key = select_candidate(comparison, CV_SELECTION_METRIC)
winner = next(
    result
    for result in evaluations
    if (result.candidate.subset, *result.candidate.order) == selected_key
)
failed_count = sum(not result.summary for result in evaluations)
print(f"{failed_count} of {len(evaluations)} candidates failed")
display(comparison)
display(pd.DataFrame(winner.fold_details))
cv_figure = cv_error_boxplots_figure(
    winner.horizons, TARGET_COLUMNS, title="Selected ARIMAX CV errors"
)
plt.show()

## Final fit and one sealed-test evaluation

Refit the winning subset/order on all eligible training rows. Coefficients and preprocessing stay fixed while newly observed levels update the state at later origins. The saved model is the training-only snapshot; rolling evaluation does not mutate it. All diagnostics use the same 24-column prediction matrix.


In [ ]:
sealed_test_started_at_ms = time_ns() // 1_000_000
final_progress = tqdm(total=2, desc="ARIMAX final fit", unit="step")
with threadpool_limits(limits=1):
    final_model, final_fit_details = fit_arimax(
        train_rows,
        contract=contract,
        candidate=winner.candidate,
        maxiter=MAXITER,
        max_outer_iter=MAX_OUTER_ITER,
        tol=GLS_TOL,
    )
final_progress.update()
final_progress.set_description("ARIMAX sealed-test forecast")

predictor_history, target_history = sealed_test_histories(dataset)
with threadpool_limits(limits=1):
    test_predictions = rolling_forecasts(
        final_model, predictor_history, target_history, test_rows["timestamp"]
    )
final_progress.update()
final_progress.close()
display(pd.DataFrame([final_fit_details]))
(
    aggregate_metrics,
    per_horizon_metrics,
    regime_definition,
    regime_aggregate_metrics,
    regime_horizon_metrics,
) = sealed_test_tables(
    test_rows[TARGET_COLUMNS],
    test_predictions,
    dataset=dataset,
    target_columns=TARGET_COLUMNS,
    station_id=TARGET_STATION_ID,
)
sealed_test_ended_at_ms = time_ns() // 1_000_000
display(aggregate_metrics)
display(per_horizon_metrics)
display(
    prediction_preview(test_rows, test_predictions, target_columns=TARGET_COLUMNS).head(
        PREDICTION_PREVIEW_ROWS
    )
)

## MLflow record and saved inference contract

The schema-6 manifest records the fitted order and coefficients, exact ordered input/output contracts, preprocessing, last fitted state time, update, future-predictor, and training-data policies, artifact hash, and execution UUID. Earlier artifacts fail validation and must be regenerated. Numerical diagnostics and CV/test results remain in MLflow.


In [ ]:
with backdated_run(
    MLFLOW_CLIENT,
    experiment_id=MLFLOW_EXPERIMENT.experiment_id,
    run_name="arimax_sealed_test",
    tags=run_tags("test", "sealed_test", NOTEBOOK_EXECUTION_UUID),
    start_time_ms=sealed_test_started_at_ms,
    end_time_ms=sealed_test_ended_at_ms,
):
    mlflow.log_params(
        {
            **COMMON_PARAMS,
            **winner.candidate.parameters(),
            **regime_mlflow_params(regime_definition),
            "scored_issue_times": len(test_rows),
            "test_start": test_rows["timestamp"].iloc[0].isoformat(),
            "test_end": test_rows["timestamp"].iloc[-1].isoformat(),
        }
    )
    mlflow.log_dict(final_fit_details, "final_fit_diagnostics.json")
    mlflow.log_table(comparison, "candidate_comparison.json")
    mlflow.log_figure(cv_figure, "selected_cv_errors.png")
    mlflow.log_metrics(
        {
            **metric_log_dict("test", aggregate_metrics, per_horizon_metrics),
            **regime_mlflow_metrics(regime_aggregate_metrics, regime_horizon_metrics),
        }
    )
    figures = {
        "test_errors.png": test_error_boxplots_figure(
            test_rows,
            test_predictions,
            per_horizon_metrics,
            TARGET_COLUMNS,
            title="ARIMAX sealed-test errors",
        ),
        "predicted_vs_actual.png": predicted_vs_actual_figure(
            test_rows[TARGET_COLUMNS],
            test_predictions,
            TARGET_COLUMNS,
            title="ARIMAX predicted vs actual",
        ),
    }
    for filename, figure in figures.items():
        mlflow.log_figure(figure, filename)
        display(figure)
        plt.close(figure)
    save_arimax_model(
        final_model,
        MODEL_PATH,
        MODEL_METADATA_PATH,
        execution_uuid=NOTEBOOK_EXECUTION_UUID,
    )
    mlflow.log_artifact(str(MODEL_PATH), artifact_path="model")
    mlflow.log_artifact(str(MODEL_METADATA_PATH), artifact_path="model")
print(f"Saved {MODEL_PATH} and {MODEL_METADATA_PATH}")

# ARIMAX saved-model review

Start here in a fresh kernel. This section reads the validated saved model and its exact MLflow execution; it does not fit candidates.


In [ ]:
import json
from pathlib import Path

import mlflow
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from IPython.display import display
from mlflow.tracking import MlflowClient
from threadpoolctl import threadpool_limits

from src.arimax import load_arimax_model, rolling_forecasts
from src.config import (
    CV_SELECTION_METRIC,
    EMBARGO_HOURS,
    FORECAST_HORIZON_HOURS,
    INITIAL_TRAIN_FRACTION,
    MLFLOW_TRACKING_URI,
    N_VALIDATION_FOLDS,
    TARGET_STATION_ID,
    WATER_LEVEL_ALARM_THRESHOLD_CM,
    WEATHER_VARIABLES,
)
from src.dataset import load_joined_dataset
from src.evaluation import load_saved_model_review, verify_saved_forecast_metrics
from src.metrics import metric_tables
from src.plots import (
    forecast_window_figures,
)
from src.regime_persistence import sealed_test_regime_tables

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
review_dir = Path("data/processed/joined")
review_model_path = Path("models") / f"arimax_{TARGET_STATION_ID}.joblib"
review_manifest_path = review_model_path.with_suffix(".json")
review_dataset = load_joined_dataset(
    review_dir / "all_stations_feature_metadata.json",
    review_dir / "all_stations_train_features.parquet",
    review_dir / "all_stations_test_features.parquet",
    station_id=TARGET_STATION_ID,
    forecast_horizon_hours=FORECAST_HORIZON_HOURS,
    weather_variables=WEATHER_VARIABLES,
    initial_train_fraction=INITIAL_TRAIN_FRACTION,
    n_validation_folds=N_VALIDATION_FOLDS,
    embargo_rows=EMBARGO_HOURS,
)
review_contract = review_dataset.contract
review_model = load_arimax_model(
    review_model_path, review_manifest_path, contract=review_contract
)
review_manifest = json.loads(review_manifest_path.read_text())
review_uuid = review_manifest.get("execution_uuid")
if not isinstance(review_uuid, str) or not review_uuid:
    raise ValueError("Saved ARIMAX manifest has no execution_uuid")
review_targets = list(review_contract.target_columns)
review_parameters = {
    "subset": review_model.candidate.subset,
    **dict(zip(("p", "d", "q"), map(str, review_model.candidate.order))),
}
if (
    tuple(review_dataset.feature_subsets[review_model.candidate.subset])
    != review_model.candidate.feature_columns
):
    raise ValueError("Saved ARIMAX subset differs from the current dataset")
review = load_saved_model_review(
    "arimax",
    review_uuid,
    input_hashes=review_dataset.input_hashes,
    forecast_horizon_hours=FORECAST_HORIZON_HOURS,
    scored_issue_times=len(review_dataset.test_rows),
    selected_parameters=review_parameters,
    quartile_cutoffs_cm=review_dataset.target_water_level_quartile_cutoffs_cm,
    quartile_reference_count=review_dataset.target_water_level_quartile_reference_count,
    alarm_threshold_cm=WATER_LEVEL_ALARM_THRESHOLD_CM,
    client=MlflowClient(),
)
print(f"Reviewing saved ARIMAX execution {review.execution_uuid}")

## Recorded CV candidates and selected test result

Failed candidates remain visible in the table and are excluded from performance plots.


In [ ]:
review_candidates = review.candidates.copy()
candidate_columns = [
    "status",
    "selected",
    *["subset", "p", "d", "q", "feature_count"],
    "cv_mae_mean",
    "cv_mae_std",
    "cv_rmse_mean",
    "cv_rmse_std",
]
print(
    f"{review_candidates['status'].eq('failed').sum()} of {len(review_candidates)} candidates failed"
)
display(review_candidates[candidate_columns])
selected_summary = {
    "execution_uuid": review.execution_uuid,
    **{
        f"cv_{metric}_mean": review.selected_candidate[f"cv_{metric}_mean"]
        for metric in ("mae", "rmse", "me", "r2")
    },
    **{
        f"test_{metric}": review.test_aggregate.iloc[0][metric]
        for metric in ("mae", "rmse", "me", "r2")
    },
}
display(pd.DataFrame([selected_summary]))
display(review.cv_horizons)
display(review.test_aggregate)
display(review.test_horizons)

successful = review_candidates.loc[review_candidates["status"].eq("ok")].copy()
for column in ("p", "d", "q"):
    successful[column] = successful[column].astype(int)
successful["order"] = successful.apply(
    lambda row: f"({row.p}, {row.d}, {row.q})", axis=1
)
for parameter in ("subset", "order"):
    figure = go.Figure()
    for value, group in successful.groupby(parameter, sort=True):
        figure.add_trace(
            go.Box(
                x=[str(value)] * len(group),
                y=group[f"cv_{CV_SELECTION_METRIC}_mean"],
                name=str(value),
                boxpoints="all",
            )
        )
    figure.update_layout(
        title=f"ARIMAX CV {CV_SELECTION_METRIC.upper()} by {parameter}",
        xaxis_title=parameter,
        yaxis_title="CV mean error (cm)",
    )
    display(figure)

## Score the saved state on the eligible sealed test cohort

The original hourly update policy is used, including explicit missing hours. Recomputed metrics must agree with the linked MLflow run.


In [ ]:
review_test_rows = review_dataset.test_rows
if (
    review_dataset.predictor_train_history.index[-1]
    >= review_dataset.predictor_test_history.index[0]
):
    raise ValueError("ARIMAX train and test histories overlap")
review_predictor_history = pd.concat(
    [review_dataset.predictor_train_history, review_dataset.predictor_test_history]
).asfreq("h")
review_target_history = pd.concat(
    [review_dataset.target_train_history, review_dataset.target_test_history]
).asfreq("h")
review_target_history["imputed"] = (
    review_target_history["imputed"].fillna(False).astype(bool)
)
with threadpool_limits(limits=1):
    review_predictions = rolling_forecasts(
        review_model,
        review_predictor_history,
        review_target_history,
        review_test_rows["timestamp"],
    )
review_aggregate, review_horizons = metric_tables(
    review_test_rows[review_targets],
    review_predictions,
    target_columns=review_targets,
    station_id=TARGET_STATION_ID,
)
_, review_regime_aggregate, review_regime_horizons = sealed_test_regime_tables(
    review_test_rows[review_targets],
    review_predictions,
    target_columns=review_targets,
    station_id=TARGET_STATION_ID,
    quartile_cutoffs_cm=review_dataset.target_water_level_quartile_cutoffs_cm,
    quartile_reference_count=review_dataset.target_water_level_quartile_reference_count,
)
verify_saved_forecast_metrics(
    review,
    review_aggregate,
    review_horizons,
    review_regime_aggregate,
    review_regime_horizons,
)
print("Saved-model aggregate, horizon, quartile, and alarm scores match MLflow")
display(review_aggregate)
display(review_horizons)

## Predictions across horizons and complete forecast windows


In [ ]:
prediction_columns = [f"prediction_{target}" for target in review_targets]
prediction_table = (
    review_test_rows[["timestamp", *review_targets]]
    .rename(columns={"timestamp": "issue_time"})
    .copy()
)
for column, values in zip(prediction_columns, review_predictions.T, strict=True):
    prediction_table[column] = values
horizons = list(range(1, len(review_targets) + 1))
labels = [f"H+{horizon:02d}" for horizon in horizons]
issue_times = pd.to_datetime(prediction_table["issue_time"], utc=True)
series = [
    {
        "x": issue_times + pd.to_timedelta(horizon, unit="h"),
        "actual": prediction_table[target],
        "prediction": prediction_table[prediction],
    }
    for horizon, target, prediction in zip(
        horizons, review_targets, prediction_columns, strict=True
    )
]
# Restyle the existing WebGL traces instead of animating frames: Plotly's
# frame animation of Scattergl traces leaves the canvas blank.
time_series = go.Figure(
    data=[
        go.Scattergl(
            x=series[0]["x"], y=series[0]["actual"], mode="lines", name="Actual"
        ),
        go.Scattergl(
            x=series[0]["x"],
            y=series[0]["prediction"],
            mode="lines",
            name="Prediction",
        ),
    ],
    layout={
        "title": "Saved ARIMAX predictions across forecast horizons",
        "xaxis_title": "Valid time",
        "yaxis_title": "Water level (cm)",
        "sliders": [
            {
                "active": 0,
                "currentvalue": {"prefix": "Forecast horizon: "},
                "steps": [
                    {
                        "label": label,
                        "method": "restyle",
                        "args": [
                            {
                                "x": [horizon_series["x"], horizon_series["x"]],
                                "y": [
                                    horizon_series["actual"],
                                    horizon_series["prediction"],
                                ],
                            },
                            [0, 1],
                        ],
                    }
                    for label, horizon_series in zip(labels, series, strict=True)
                ],
            }
        ],
    },
)
display(time_series)
windows = forecast_window_figures(
    prediction_table,
    review_dataset.target_context_series,
    water_level_column=f"{TARGET_STATION_ID}__water_level",
    imputed_column=f"{TARGET_STATION_ID}__imputed",
    prediction_columns=prediction_columns,
    target_columns=review_targets,
    horizons=horizons,
    label_prefix="ARIMAX",
)
display(windows["best"])
display(windows["worst"])

## Absolute and signed error distributions


In [ ]:
signed_errors = review_predictions - review_test_rows[review_targets].to_numpy(
    dtype=float
)
absolute_errors = np.abs(signed_errors)
absolute_figure = go.Figure(
    data=[
        go.Box(
            x=labels * len(absolute_errors),
            y=absolute_errors.reshape(-1),
            name="Absolute error",
        ),
        go.Scatter(x=labels, y=review_horizons["mae"], mode="markers", name="MAE"),
        go.Scatter(x=labels, y=review_horizons["rmse"], mode="markers", name="RMSE"),
    ]
)
absolute_figure.update_layout(
    title="Saved ARIMAX absolute errors", xaxis_title="Horizon", yaxis_title="cm"
)
display(absolute_figure)
signed_figure = go.Figure(
    data=[
        go.Box(
            x=labels * len(signed_errors),
            y=signed_errors.reshape(-1),
            name="Signed error",
        ),
        go.Scatter(x=labels, y=signed_errors.mean(axis=0), mode="markers", name="ME"),
    ]
)
signed_figure.add_hline(y=0, line_dash="dash")
signed_figure.update_layout(
    title="Saved ARIMAX signed errors",
    xaxis_title="Horizon",
    yaxis_title="Prediction minus actual (cm)",
)
display(signed_figure)

## ARIMAX model and fit diagnostics


In [ ]:
coefficient_names = (
    ["intercept"] if review_model.candidate.order[1] == 0 else []
) + list(review_model.candidate.feature_columns)
display(
    pd.DataFrame(
        {"coefficient": coefficient_names, "scaled_value": review_model.coefficients}
    )
)
display(review_model.result.summary())
fit_diagnostics = mlflow.artifacts.load_dict(
    f"runs:/{review.sealed_run_id}/final_fit_diagnostics.json"
)
if not isinstance(fit_diagnostics, dict) or not fit_diagnostics:
    raise ValueError("ARIMAX sealed_test is missing final fit diagnostics")
display(pd.DataFrame([fit_diagnostics]))

## Quartile and alarm diagnostics


In [ ]:
display(review.regime_aggregate)
display(review.regime_horizons)
for regime_group in ("quartile", "alarm"):
    display(
        regime_aggregate_figure(
            review.regime_aggregate, regime_group=regime_group, model_label="ARIMAX"
        )
    )
    for metric in ("mae", "rmse", "me"):
        display(
            regime_horizon_figure(
                review.regime_horizons,
                regime_group=regime_group,
                metric=metric,
                model_label="ARIMAX",
            )
        )